In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Practica1_RDD") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark funcionando correctamente")
print("Versión de Spark:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/10 17:07:58 WARN Utils: Your hostname, MacBook-Air-de-Carmen-2.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.3 instead (on interface en0)
26/11/10 17:07:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/10 17:07:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark funcionando correctamente
Versión de Spark: 4.2.0


In [2]:
# EJERCICIO 1
# Leer el fichero alturas_v0.csv como un RDD

alturas = sc.textFile("alturas_v0.csv")

# Mostramos los primeros 10 datos
alturas.take(10)

['173', '161', '176', '180', '1.62', '176', '166', '174', '-100', '158']

In [3]:
# Convertimos los datos de texto a números
alturas_float = alturas.map(lambda x: float(x))

# Eliminamos los datos erróneos (-100)
alturas_filtradas = alturas_float.filter(lambda x: x != -100)

# Convertimos a centímetros las alturas que están en metros
alturas_limpias = alturas_filtradas.map(
    lambda x: x * 100 if x < 3 else x
)

# Comprobamos los primeros 10 resultados
alturas_limpias.take(10)

[173.0, 161.0, 176.0, 180.0, 162.0, 176.0, 166.0, 174.0, 158.0, 162.0]

In [4]:
# Calculamos la media manualmente

# Número total de alturas válidas
n = alturas_limpias.count()

# Sumamos todas las alturas usando reduce
suma = alturas_limpias.reduce(lambda x, y: x + y)

# Calculamos la media
media = suma / n

print("Número de datos válidos:", n)
print("Suma de las alturas:", suma)
print("Media:", media)

Número de datos válidos: 1911
Suma de las alturas: 322841.0
Media: 168.93825222396651


In [5]:
# Calculamos la desviación típica manualmente

# Calculamos para cada altura (altura - media)^2
diferencias_cuadrado = alturas_limpias.map(
    lambda x: (x - media) ** 2
)

# Sumamos todas las diferencias al cuadrado
suma_diferencias = diferencias_cuadrado.reduce(lambda x, y: x + y)

# Calculamos la varianza
varianza = suma_diferencias / n

# Calculamos la desviación típica
desviacion_tipica = varianza ** 0.5

print("Varianza:", varianza)
print("Desviación típica:", desviacion_tipica)

Varianza: 61.372430016969126
Desviación típica: 7.834055783371033


In [6]:
# EJERCICIO 2
# Leer el fichero alturas.csv como un RDD

datos = sc.textFile("alturas.csv")

# Mostramos los primeros 10 registros
datos.take(10)

['H,173',
 'M,161',
 'H,176',
 'H,180',
 'M,1.62',
 'H,176',
 'M,166',
 'H,174',
 'M,-100',
 'M,158']

In [7]:
# Separamos cada línea en género y altura
datos_separados = datos.map(
    lambda x: (x.split(",")[0], float(x.split(",")[1]))
)

# Eliminamos los datos erróneos (-100)
datos_filtrados = datos_separados.filter(
    lambda x: x[1] != -100
)

# Convertimos a centímetros las alturas que están en metros
datos_limpios = datos_filtrados.map(
    lambda x: (x[0], x[1] * 100 if x[1] < 3 else x[1])
)

# Comprobamos los primeros 10
datos_limpios.take(10)

[('H', 173.0),
 ('M', 161.0),
 ('H', 176.0),
 ('H', 180.0),
 ('M', 162.0),
 ('H', 176.0),
 ('M', 166.0),
 ('H', 174.0),
 ('M', 158.0),
 ('M', 162.0)]

In [8]:
# Separamos las alturas por género

hombres = datos_limpios.filter(lambda x: x[0] == "H").map(lambda x: x[1])
mujeres = datos_limpios.filter(lambda x: x[0] == "M").map(lambda x: x[1])

# Número de datos de cada género
n_hombres = hombres.count()
n_mujeres = mujeres.count()

# Suma de las alturas de cada género
suma_hombres = hombres.reduce(lambda x, y: x + y)
suma_mujeres = mujeres.reduce(lambda x, y: x + y)

# Calculamos las medias
media_hombres = suma_hombres / n_hombres
media_mujeres = suma_mujeres / n_mujeres

print("Número de hombres:", n_hombres)
print("Media hombres:", media_hombres)

print("Número de mujeres:", n_mujeres)
print("Media mujeres:", media_mujeres)

Número de hombres: 981
Media hombres: 173.46075433231397
Número de mujeres: 930
Media mujeres: 164.16774193548386


In [9]:
# Calculamos la desviación típica de los hombres

dif_cuadrado_hombres = hombres.map(
    lambda x: (x - media_hombres) ** 2
)

suma_dif_hombres = dif_cuadrado_hombres.reduce(lambda x, y: x + y)

varianza_hombres = suma_dif_hombres / n_hombres

desviacion_hombres = varianza_hombres ** 0.5


# Calculamos la desviación típica de las mujeres

dif_cuadrado_mujeres = mujeres.map(
    lambda x: (x - media_mujeres) ** 2
)

suma_dif_mujeres = dif_cuadrado_mujeres.reduce(lambda x, y: x + y)

varianza_mujeres = suma_dif_mujeres / n_mujeres

desviacion_mujeres = varianza_mujeres ** 0.5


print("Desviación típica hombres:", desviacion_hombres)
print("Desviación típica mujeres:", desviacion_mujeres)

Desviación típica hombres: 7.5312288366375055
Desviación típica mujeres: 4.684891325217869
